## Análise de dados utilizando PySpark no Databricks
- Uploads de arquivos para análise
- Importação de bibliotecas necessárias
- Listagem de arquivos
- Criação do DataFrame e juntando todos em um DF único
- Criação de novas colunas
- Verificação de valores nulos
- Criação de novas colunas
- Verificação de valores nulos
- Remoção de linhas duplicadas
- Remoção de colunas
- Remoção de valores nulos
- Alteração de tipo de coluna
- Renomear coluna(s)
- Renomear colunas utilizando dicionário
- Criação de coluna condicional
- Ajuste de ordem de exibição de colunas

- Início de análise

- Agregar valores (groupBy)
- Análise de vendas (vendas por cidade)
- Ticket médio por cidade
- Produto com maior número de vendas
- Quais produtos vendem mais combinados? (produto1 + produto2)

In [0]:
# Importando as bibliotecas
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# Utilizando DBUtils para descobrir quais arquivos vamos importar para o DataFrame
arquivos = dbutils.fs.ls("/Volumes/workspace/default/datadev-engineering")
arquivos

In [0]:
# Criando lista com os arquivos de 2019
arquivos2019 = [arquivo.path for arquivo in arquivos if arquivo.path.endswith('2019.csv')]
arquivos2019


In [0]:
# Criando primeiro DF
df = spark.read.csv(arquivos2019[0],inferSchema=True,header=True)
display(df)

In [0]:
# Criando outros DF e juntando com DF original
for mes in arquivos2019[1:]:
    df_temporario = spark.read.csv(mes, inferSchema=True, header=True)
    df = df.union(df_temporario)
    print(f'{mes} incluido com sucesso')


In [0]:
# Conferindo se deu tudo certo com o Union
df.filter(col('Order Date').startswith('08')).show()

In [0]:
# Criar novas colunas (mes, dia, hora)
df = df.withColumn('Month', substring(col('Order Date'), 0,2))\
  .withColumn('Day',substring(col('Order Date'),4,2))\
  .withColumn('Hour',substring(col('Order Date'),-5,2))\
  .withColumn('Minute', substring(col('Order Date'),-2,2))

In [0]:
# Verificando valores nulos
for coluna in df.columns:
    print(coluna, df.filter(col(coluna).isNull()).count())

In [0]:
# Verificar quais as columas com valores nulos 
df = df.filter(col('Order ID').isNotNull())

In [0]:
# Filtrar DF e deixar apenas valores nulos
df = df.dropDuplicates()

In [0]:

# Excluindo coluna de Order Date
df = df.drop(col('Order Date'))

In [0]:
# Alterando tipo de colunas (Month, Day, Hour, Price Each, Order ID)
df = df.withColumn('Month', col('Month').cast(IntegerType()))

In [0]:
# Renomeando a coluna Order ID
df = df.withColumnRenamed('Order ID','ID')

In [0]:
# Renomeando mais de 1 coluna de uma vez
colunas_renomeadas = {
    'Product':'Produto',
    'Quantity Ordered':'Qtd',
    'Price Each':'Valor_Unitario',
    'Purchase Address':'Endereco',
    'Month':'Mes',
    'Day':'Dia',
    'Hour':'Hora',
    'Minute':'Minuto',
}
df = df.withColumnsRenamed(colunas_renomeadas)

In [0]:
df = df.withColumn('Total',col('Qtd')*col('Valor_Unitario'))

In [0]:
# Ordenando as colunas
df = df.select(
    [
        'ID',
        'Produto',
        'Qtd',
        'Valor_Unitario',
        'Total',
        'Endereco',
        'Mes',
        'Dia',
        'Hora',
        'Minuto' 
    ]
)

In [0]:
df.show()